# Módulo 1 — Referencias poblacionales de envejecimiento

Atributos clásicos, ICC test-retest y curvas longitudinales TRAIN. Este cuaderno no entrena difusión ni genera imágenes. FFHQ-Aging es un proxy del dominio de entrenamiento, no una etiqueta de nacionalidad. `sample_kaggle` no es FFHQ-Aging.


In [ ]:
# Ejecuta manualmente en Colab, desde la raíz del repositorio.
%pip install -e ".[aging-bias]"


In [ ]:
from pathlib import Path
from dataclasses import replace
import json
import pandas as pd
from src.aging_bias import ReferenceConfig, FeatureConfig, build_reference_bundle, describe_population_aging, run_self_checks
from src.comparisong.evaluation_backends_v2 import prepare_reference_backend, prepare_embedding_reference_backend

COLOMBIA_TRAIN = Path("/content/drive/MyDrive/dataset_def/train")
FAIRFACE_ROOT = None  # Opcional: Path("/content/drive/MyDrive/FairFace_Frontal")
FFHQ_AGING = Path("/content/drive/MyDrive/FFHQ-Aging")
TEST_MANIFEST = Path("/content/drive/MyDrive/respaldo_test_comparison/transitions.csv")
PREVIOUS_REFERENCE_DIR = None  # Opcional: Path("/content/drive/MyDrive/aging_bias_references"); solo lectura
REFERENCE_DIR = Path("/content/drive/MyDrive/aging_bias_references_v2")  # Carpeta NUEVA
ASSETS = Path("/content/evaluation_models_v2")


## 1. Validación matemática, sin redes

Se detiene si la regresión o los pesos no cumplen las anclas sintéticas.


In [ ]:
checks = run_self_checks(REFERENCE_DIR / "validation")
display(checks)
assert checks.passed.all()


## 2. Preparar solo landmarks (CPU)

Descarga explícita al ejecutar esta celda: un FaceLandmarker task pequeño. No carga FairFace, AdaFace, DEX ni Inception. Archivos existentes se reutilizan.


In [ ]:
reference_backend = prepare_reference_backend(
    ASSETS, allow_download=True,
    feature_config=FeatureConfig(
        crop_size=256, harmonize_sigma=1.0,
        max_abs_yaw=15.0, max_smile_score=0.25, max_jaw_open_score=0.15,
        normalize_color_illumination=True,
    ),
)


## 3. Configuración sin fuga de test

Usa una carpeta que contenga **solo identidades de train**, con `id_x/edad[_repeticion].jpg`. Si usas la raíz completa, cambia `train_only=False` y proporciona `split_path` con columnas `person_id,split`. El test se utiliza exclusivamente para excluir IDs, nunca para ajustar referencias.

FFHQ requiere imágenes locales y `ffhq_aging_labels.csv` oficial. No se sustituye por Kaggle/FG-NET. Fotos de edades cercanas pero distintas son necesarias para ICC; sin ellas no se inventa confiabilidad.

Suplemento opcional: cambia solo `FAIRFACE_ROOT` en la celda de rutas. Espera fotos en esa carpeta (o `images/`), `age_predictions.csv` con `filename,age_pred` y, si existe, usa `metadata_frontal.csv/json` como lista permitida. Ignora rutas antiguas del CSV y archivos ausentes. Usa el mismo preprocesamiento y QC; cada foto recibe una identidad independiente: contribuye a normalización, curva transversal y PCA, NO aporta pares ICC ni información longitudinal. Guarda `population_source` y `age_source` en los parquet. Las edades predichas no son edades cronológicas y la etiqueta Latino_Hispanic no equivale a nacionalidad colombiana; compara también referencias sin este suplemento para comprobar sensibilidad.


In [ ]:
reference_config = ReferenceConfig(
    colombia_root=COLOMBIA_TRAIN, ffhq_aging_root=FFHQ_AGING,
    output_dir=REFERENCE_DIR, train_only=True,
    module1_close=True, previous_reference_dir=PREVIOUS_REFERENCE_DIR,
    module1_assets_root=ASSETS, module1_allow_download=True,
    fairface_root=FAIRFACE_ROOT,
    test_manifest_path=TEST_MANIFEST,
    # split_path="/content/drive/MyDrive/dataset_def/splits.csv",
    features=reference_backend["feature_config"],
    ffhq_min_conf=0.6, ffhq_max_per_group=500,
    # Política predefinida: no ajustar umbrales después de mirar el test.
    retest_min_age=30, retest_max_expression_delta=0.10,
    min_icc=0.30, min_homog_icc=0.50, weight_attributes_by_icc=True,
    min_retest_pairs=10, seed=2026, resume=True,
)


In [ ]:
from src.aging_bias.references import load_colombia_manifest
from itertools import combinations

train = load_colombia_manifest(reference_config)
adult_train = train[train.age >= reference_config.retest_min_age]
near_pairs = sum(abs(float(a)-float(b)) <= 1 for _, g in adult_train.groupby("identity_id") for a,b in combinations(g.age,2))
print(f"Train: {train.identity_id.nunique()} identidades, {len(train)} imágenes; {near_pairs} pares adultos potenciales")
assert near_pairs >= reference_config.min_retest_pairs, "Faltan retests adultos; no se sustituirán por jóvenes. El conteo definitivo exige fotos distintas y QC."


In [ ]:
references = build_reference_bundle(
    reference_config, feature_backend=reference_backend["feature_backend"],
    landmark_backend=reference_backend["landmark_backend"],
)
display(pd.read_csv(REFERENCE_DIR / "attribute_reliability.csv"))
display(pd.read_csv(REFERENCE_DIR / "reference_quality.csv"))
print("Atributos activos:", references.attributes)


## 4. ¿Las poblaciones envejecen distinto?

Bootstrap por identidad en Colombia y por imagen dentro del grupo de edad en FFHQ. Revisa soporte, intervalos e identificación antes de evaluar modelos. El problema edad-período-cohorte no desaparece por compartir referencias.


In [ ]:
# El cierre ya calculó bootstrap/diagnósticos; aquí solo los mostramos.
population_table = pd.read_csv(REFERENCE_DIR / "population_contrasts.csv")
population_table.attrs["identification"] = json.loads((REFERENCE_DIR / "identification.json").read_text())
display(population_table)
display(pd.read_csv(REFERENCE_DIR / "capture_sensitivity.csv"))
print(population_table.attrs["identification"])
from IPython.display import Image, display
display(Image(filename=str(REFERENCE_DIR / "population_aging.png")))
display(pd.read_csv(REFERENCE_DIR / "diagnostics/icc_tta_comparison.csv"))
display(pd.read_csv(REFERENCE_DIR / "tables/zone_consistency.csv"))
display(pd.read_csv(REFERENCE_DIR / "diagnostics/resolution_matching.csv"))
print("Figuras y control de landmarks:", REFERENCE_DIR / "figures")
print("Placebo interno (no paper):", REFERENCE_DIR / "internal/placebo_ffhq.csv")


## 5. Robustez DINOv2 (GPU si hay CUDA, referencias separadas)

Ejecuta las siguientes celdas para comparar la fiabilidad de ambos espacios. Descarga explícita de DINOv2-small solo al ejecutar; PCA de 16 dimensiones sobre un máximo de 2.000 crops de train + suplemento opcional + proxy, nunca test. Un único encoder residente en GPU si hay CUDA, lotes acotados y caché de vectores/PCA; PCA en CPU. No se garantiza ICC alto: se vuelve a medir.


In [ ]:
embedding_config = replace(
    reference_config, output_dir=REFERENCE_DIR / "dinov2", feature_backend="embedding",
    embedding_components=16, embedding_fit_max_images=2000, embedding_batch_size=16,
)
embedding_backend = prepare_embedding_reference_backend(
    embedding_config, root=ASSETS,
    device="auto",  # CUDA si está disponible; no mueve el encoder por imagen.
    landmark_backend=reference_backend["landmark_backend"], allow_download=True,
)
embedding_config.features = embedding_backend["feature_config"]  # Mismo target IOD para PCA/extracción
embedding_references = build_reference_bundle(
    embedding_config, feature_backend=embedding_backend["feature_backend"],
    landmark_backend=embedding_backend["landmark_backend"],
)
display(pd.read_csv(embedding_config.output_dir / "attribute_reliability.csv"))
embedding_population = pd.read_csv(embedding_config.output_dir / "population_contrasts.csv")
display(embedding_population)


## Interpretación y salidas

`attribute_reliability.csv` conserva ICC adulto y de todas las edades, n de pares/n de identidades, elegibilidad para pi/homogeneización y pesos. El ICC adulto no valida niños. `reference_quality.csv` muestra exclusiones; `capture_sensitivity.csv` compara el mismo panel con/sin blanco y negro, resolución y desenfoque y marca cambios de signo. Estos controles no resuelven causalmente edad-período-cohorte.

La corrección de iluminación es local: conserva cromaticidad y no blanquea piel. ITA/canas siguen siendo exploratorios sin calibración global o segmentación validada. La ponderación ICC no corrige errores en regresores.

El módulo 2 lee la política de estas referencias sin cambiar la llamada: pi usa atributos con ICC ≥0.30 ponderados por ICC; la homogeneización usa ICC ≥0.50 y la tabla principal muestra `homog_excess` respecto al oráculo del mismo panel, con bootstrap pareado. El valor crudo queda para diagnóstico. Si el oráculo no se identifica, el exceso es NaN, no cero.

El cierre se guarda SOLO en la carpeta nueva. Revisa `figures/malar_landmarks_check.png` antes de interpretar los pómulos. Las edades de FairFace siguen siendo las de tu CSV. Solo FFHQ 50–69 recibe estimación adicional para las figuras (guardada aparte, sin cambiar las referencias oficiales). Las figuras son descriptivas; la evidencia está en `tables/zone_consistency.csv` y sus contrastes con IC. El placebo queda exclusivamente en `internal/`.

Conserva toda la carpeta, incluidos `references.npz/json`, `embedding_pca.npz/json`, cachés y reportes. Al cambiar controles se invalida la caché de atributos automáticamente: no reutiliza mediciones incompatibles ni regenera imágenes de difusión. Comparar pi entre backends requiere ejecutar módulo 2 con sus referencias y su backend correspondiente; estas celdas solo construyen referencias.
